# 08. 야코비안과 미분기구학

야코비안은 **관절 속도 → 엔드이펙터 속도** 의 선형 매핑이다.

$$\dot{x} = J(\theta)\,\dot{\theta}$$

- $\dot{\theta} \in \mathbb{R}^n$ : 관절 속도 (n = DoF)
- $\dot{x} \in \mathbb{R}^6$ : EE 속도 (선속도 3 + 각속도 3)
- $J \in \mathbb{R}^{6 \times n}$ : 야코비안 행렬 (자세에 따라 달라짐)

**왜 중요한가:**
- $\det(J) = 0$ → **특이 자세** → 특정 방향으로 움직일 수 없음
- $J^+$ (의사역행렬) → 역기구학 풀기
- $J^T$ → 힘/토크 변환: $\tau = J^T f$
- 조작 가능도 $= \sqrt{\det(J J^T)}$ → 자세 품질 지표

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
os.makedirs('assets', exist_ok=True)

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False

L1, L2 = 1.0, 0.8

def fk_2link(t1_deg, t2_deg):
    t1 = np.radians(t1_deg); t2 = np.radians(t2_deg)
    x = L1*np.cos(t1) + L2*np.cos(t1+t2)
    y = L1*np.sin(t1) + L2*np.sin(t1+t2)
    return np.array([x, y])

def jacobian_2link(t1_deg, t2_deg):
    t1 = np.radians(t1_deg); t2 = np.radians(t2_deg)
    # dx/dtheta, dy/dtheta — 해석적으로 유도
    J = np.array([
        [-L1*np.sin(t1) - L2*np.sin(t1+t2),  -L2*np.sin(t1+t2)],
        [ L1*np.cos(t1) + L2*np.cos(t1+t2),   L2*np.cos(t1+t2)]
    ])
    return J

## 1. 야코비안 유도 — 2R 팔

엔드이펙터 위치:
$$x = L_1\cos\theta_1 + L_2\cos(\theta_1+\theta_2)$$
$$y = L_1\sin\theta_1 + L_2\sin(\theta_1+\theta_2)$$

편미분하면:
$$J = \frac{\partial(x,y)}{\partial(\theta_1,\theta_2)} = \begin{bmatrix} -L_1 s_1 - L_2 s_{12} & -L_2 s_{12} \\ L_1 c_1 + L_2 c_{12} & L_2 c_{12} \end{bmatrix}$$

수치 미분으로 검증할 수 있다.

In [ ]:
def jacobian_numerical(t1_deg, t2_deg, eps=1e-6):
    J = np.zeros((2, 2))
    for i, dt in enumerate([(eps,0),(0,eps)]):
        p_plus  = fk_2link(t1_deg + np.degrees(dt[0]), t2_deg + np.degrees(dt[1]))
        p_minus = fk_2link(t1_deg - np.degrees(dt[0]), t2_deg - np.degrees(dt[1]))
        J[:, i] = (p_plus - p_minus) / (2*eps)
    return J

t1, t2 = 45.0, 30.0
J_analytical = jacobian_2link(t1, t2)
J_numerical  = jacobian_numerical(t1, t2)

print(f'자세: θ1={t1}°, θ2={t2}°')
print(f'\n해석 야코비안:\n{J_analytical.round(6)}')
print(f'\n수치 야코비안:\n{J_numerical.round(6)}')
print(f'\n최대 오차: {np.max(np.abs(J_analytical - J_numerical)):.2e}')

# 야코비안의 의미 시각화
fig, ax = plt.subplots(figsize=(8, 8))

ee = fk_2link(t1, t2)
j1 = np.array([L1*np.cos(np.radians(t1)), L1*np.sin(np.radians(t1))])
ax.plot([0, j1[0]], [0, j1[1]], 'o-', color='#534AB7', lw=3, markersize=10)
ax.plot([j1[0], ee[0]], [j1[1], ee[1]], 'o-', color='#1D9E75', lw=3, markersize=10)
ax.plot(*ee, '*', color='#E85D24', markersize=15, zorder=5)

# 관절 속도 → EE 속도 열벡터 시각화
scale = 0.4
for i, (color, label) in enumerate(zip(['#534AB7','#1D9E75'],
                                        ['dθ₁=1 → EE 속도', 'dθ₂=1 → EE 속도'])):
    col = J_analytical[:, i] * scale
    ax.annotate('', xy=ee+col, xytext=ee,
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(*(ee + col*1.2), label, color=color, fontsize=9, fontweight='bold')

ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title(f'야코비안 열벡터 = 각 관절 단위 속도가 만드는 EE 속도\n(θ1={t1}°, θ2={t2}°)', fontsize=11)
plt.tight_layout()
plt.savefig('assets/08_jacobian_cols.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 특이 자세 (Singularity) — $\det(J) = 0$

2R 팔에서:
$$\det(J) = L_1 L_2 \sin\theta_2$$

$\theta_2 = 0°$ 또는 $180°$ 일 때 특이 자세 — 팔이 완전히 펴지거나 접힌 상태.
이 순간 야코비안이 랭크를 잃고 특정 방향 이동이 불가능해진다.

In [ ]:
# 작업 공간 전체에서 det(J) 계산
t1_range = np.linspace(-180, 180, 200)
t2_range = np.linspace(-180, 180, 200)
T1, T2 = np.meshgrid(t1_range, t2_range)

det_J = L1 * L2 * np.sin(np.radians(T2))   # 2R 팔 해석 공식
manip = np.abs(det_J)   # 조작 가능도

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 조작 가능도 히트맵
im = axes[0].contourf(T1, T2, manip, levels=30, cmap='RdYlGn')
axes[0].contour(T1, T2, manip, levels=[0.05], colors='red', linewidths=2)
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('θ1 (도)'); axes[0].set_ylabel('θ2 (도)')
axes[0].set_title('조작 가능도 |det(J)|\n빨간선 = 특이 자세 근처', fontsize=11)
axes[0].axhline(0, color='white', lw=1.5, linestyle='--')
axes[0].axhline(180, color='white', lw=1.5, linestyle='--')
axes[0].axhline(-180, color='white', lw=1.5, linestyle='--')

# 특이 자세 vs 일반 자세 비교
ax2 = axes[1]
configs_sing = [
    (45,  0,   True,  '특이 자세 θ2=0° (완전히 펴짐)'),
    (45,  45,  False, '일반 자세 θ2=45°'),
    (90,  180, True,  '특이 자세 θ2=180° (완전히 접힘)'),
]
colors_s = ['#E85D24','#1D9E75','#BA7517']

for (t1s, t2s, is_sing, lbl), color in zip(configs_sing, colors_s):
    ee_s = fk_2link(t1s, t2s)
    j1_s = np.array([L1*np.cos(np.radians(t1s)), L1*np.sin(np.radians(t1s))])
    lw = 3 if is_sing else 2
    ls = '--' if is_sing else '-'
    ax2.plot([0, j1_s[0], ee_s[0]], [0, j1_s[1], ee_s[1]],
             linestyle=ls, color=color, lw=lw, marker='o', markersize=7,
             label=lbl)

ax2.set_xlim(-2.2, 2.2); ax2.set_ylim(-2.2, 2.2)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)
ax2.set_title('특이 자세 vs 일반 자세\n점선 = 특이 자세', fontsize=11)
ax2.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('assets/08_singularity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'det(J) @ θ2=0°:  {L1*L2*np.sin(0):.4f}  ← 특이')
print(f'det(J) @ θ2=45°: {L1*L2*np.sin(np.radians(45)):.4f}')
print(f'det(J) @ θ2=90°: {L1*L2*np.sin(np.radians(90)):.4f}  ← 최대 조작 가능도')

## 3. 의사역행렬 역기구학 — $\dot{\theta} = J^+ \dot{x}$

역기구학: 원하는 EE 속도 $\dot{x}$ 가 주어졌을 때 필요한 관절 속도 $\dot{\theta}$ 계산.

$$\dot{\theta} = J^+ \dot{x} = J^T(J J^T)^{-1} \dot{x} \quad \text{(2×2 경우 } J^+ = J^{-1}\text{)}$$

특이 자세 근처에서 $J^{-1}$ 가 폭발하는 걸 막으려면 **댐핑 최소제곱** 을 쓴다:

$$\dot{\theta} = J^T(J J^T + \lambda^2 I)^{-1} \dot{x}$$

In [ ]:
def ik_step(t1_deg, t2_deg, dx, lam=0.0):
    J = jacobian_2link(t1_deg, t2_deg)
    if lam == 0:
        # 일반 역행렬 (2×2 이므로 가능)
        try:
            dtheta = np.linalg.solve(J, dx)
        except np.linalg.LinAlgError:
            dtheta = np.zeros(2)
    else:
        # 댐핑 최소제곱
        JJT = J @ J.T
        dtheta = J.T @ np.linalg.solve(JJT + lam**2 * np.eye(2), dx)
    return np.degrees(dtheta)

# 목표 궤적 추종 시뮬레이션
np.random.seed(0)
dt = 0.05
steps = 120

# 원형 궤적
t_traj = np.linspace(0, 2*np.pi, steps)
cx, cy, r_traj = 0.8, 0.6, 0.4
x_target = cx + r_traj*np.cos(t_traj)
y_target = cy + r_traj*np.sin(t_traj)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, lam, title in [
    (axes[0], 0.0,  '일반 역기구학 (λ=0)'),
    (axes[1], 0.05, '댐핑 역기구학 (λ=0.05)'),
]:
    t1, t2 = 45.0, 30.0
    traj_ee = [fk_2link(t1, t2)]
    dthetas_norm = []

    for i in range(steps - 1):
        ee = fk_2link(t1, t2)
        dx = (np.array([x_target[i+1], y_target[i+1]]) - ee) / dt * dt
        dth = ik_step(t1, t2, dx, lam=lam)
        # 속도 제한
        dth = np.clip(dth, -15, 15)
        t1 += dth[0]; t2 += dth[1]
        traj_ee.append(fk_2link(t1, t2))
        dthetas_norm.append(np.linalg.norm(dth))

    traj_ee = np.array(traj_ee)
    ax.plot(x_target, y_target, 'k--', lw=1.5, alpha=0.5, label='목표 궤적')
    ax.plot(traj_ee[:,0], traj_ee[:,1], color='#E85D24', lw=2, label='실제 궤적')
    ax.scatter(x_target[0], y_target[0], s=80, color='#534AB7', zorder=5, label='시작점')

    err = np.mean(np.linalg.norm(traj_ee[1:] - np.stack([x_target[1:],y_target[1:]]).T, axis=1))
    ax.set_xlim(-0.2, 2.0); ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}\n평균 추적 오차: {err:.4f}m', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

plt.suptitle('야코비안 역기구학으로 원형 궤적 추종', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('assets/08_ik_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 조작 가능도 타원체 (Manipulability Ellipsoid)

$J J^T$ 의 고유벡터·고유값으로 각 방향의 속도 증폭비를 시각화.

타원체가 클수록 좋은 자세 — 모든 방향으로 움직이기 쉬움.
특이 자세에선 타원체가 선으로 퇴화.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

t1_vis = np.linspace(0, 360, 8, endpoint=False)
t2_vis = [90, 60, 120, 30, 150, 45, 90, 90]

for t1s, t2s in zip(t1_vis, t2_vis):
    ee = fk_2link(t1s, t2s)
    J = jacobian_2link(t1s, t2s)
    JJT = J @ J.T
    evals, evecs = np.linalg.eigh(JJT)
    evals = np.maximum(evals, 0)  # 수치 오차 방지

    scale = 0.18
    theta_ell = np.linspace(0, 2*np.pi, 60)
    ell_local = np.array([np.sqrt(evals[0])*np.cos(theta_ell),
                          np.sqrt(evals[1])*np.sin(theta_ell)]) * scale
    ell_world = evecs @ ell_local + ee[:, None]

    manip_val = np.sqrt(max(np.linalg.det(JJT), 0))
    color = plt.cm.RdYlGn(min(manip_val / 0.8, 1.0))
    ax.fill(ell_world[0], ell_world[1], alpha=0.35, color=color)
    ax.plot(ell_world[0], ell_world[1], color=color, lw=1.2)
    ax.plot(*ee, 'k.', markersize=5)

# 링크 하나만 예시로
t1_ex, t2_ex = 90.0, 90.0
ee_ex = fk_2link(t1_ex, t2_ex)
j1_ex = np.array([L1*np.cos(np.radians(t1_ex)), L1*np.sin(np.radians(t1_ex))])
ax.plot([0, j1_ex[0], ee_ex[0]], [0, j1_ex[1], ee_ex[1]],
        'o--', color='gray', lw=1.5, markersize=6, alpha=0.5)

# 컬러바 대용 그라디언트
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
sm = ScalarMappable(cmap='RdYlGn', norm=Normalize(0, 0.8))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='조작 가능도 √det(JJᵀ)', shrink=0.7)

ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title('조작 가능도 타원체\n타원이 클수록 좋은 자세 / 선이면 특이 자세', fontsize=12)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.tight_layout()
plt.savefig('assets/08_manipulability.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| 야코비안 | $\dot{x} = J\dot{\theta}$ | 속도 기구학 |
| 특이 자세 | $\det(J) = 0$ | 회피 경로 계획 |
| 역기구학 | $\dot{\theta} = J^+\dot{x}$ | 궤적 추종 제어 |
| 댐핑 최소제곱 | $(JJ^T + \lambda^2 I)^{-1}$ | 특이 자세 근처 안정화 |
| 조작 가능도 | $\sqrt{\det(JJ^T)}$ | 자세 품질 평가 |

---

**선형대수 파트 완료.**
다음 단계: `02_calculus/` — 미적분, 그라디언트, 헤시안